# Three small OCR models, one page

Every OCR model here has its own prompt protocol, and that is the most common
reason a working model looks broken. All three fit well inside 16 GB.

| model | protocol | prompts |
|---|---|---|
| GLM-OCR | task prefix, templated | `Text Recognition:`, `Formula Recognition:`, `Table Recognition:` |
| PaddleOCR-VL 1.6 | task prefix, templated | `OCR:`, `Table Recognition:`, `Chart Recognition:` |
| Falcon-OCR | **bare category, no chat template** | `plain`, `text`, `table`, `formula`, `title` |

Use `tiiuae/Falcon-OCR`, not `mlx-community/Falcon-OCR-bf16`: the converted repo
returns incoherent tokens for every prompt, while the original is clean through
the identical code path.

On-disk sizes: GLM-OCR 1.2 GB, PaddleOCR-VL 0.7 GB, Falcon-OCR 1.0 GB. This
notebook does not report peak memory per model, because peak is a property of
the process — every model stays resident after it is loaded, so the second
model's "peak" is really the first model's. Measure that one model per process.

In [1]:
from mlx_vlm import apply_chat_template, generate, load

PAGE = "../images/paper.png"
FORMULA = "../images/latex.png"
TABLE = "../images/menu.webp"


def run(model_id, prompt, image, max_tokens=120, templated=True):
    model, processor = load(model_id)
    text = (
        apply_chat_template(processor, model.config, prompt, num_images=1)
        if templated
        else prompt
    )
    result = generate(
        model, processor, text, image=[image], max_tokens=max_tokens, temperature=0.0
    )
    print(f"{model_id.split('/')[-1]}  ·  {prompt!r}")
    print(result.text.strip())
    print(f"[{result.generation_tps:.1f} tok/s]")

## Plain text

The third call passes a bare category and skips `apply_chat_template` entirely.

In [2]:
run("mlx-community/GLM-OCR-4bit", "Text Recognition:", PAGE)
run("mlx-community/PaddleOCR-VL-1.6-4bit", "OCR:", PAGE)
run("tiiuae/Falcon-OCR", "plain", PAGE, templated=False)

GLM-OCR-4bit  ·  'Text Recognition:'
Reducing Transformer Key-Value Cache Size with Cross-Layer Attention

William Brandon*  
MIT CSAIL  
wbrandon@csail.mit.edu

Mayank Mishra*  
MIT-IBM Watson AI Lab

Aniruddha Nrusimha  
MIT CSAIL

Rameswar Panda  
MIT-IBM Watson AI Lab

Jonathan Ragan-Kelley  
MIT CSAIL

Abstract

Key-value (KV) caching plays an essential role in accelerating decoding for transformer-based autoregressive
[515.9 tok/s]


/Users/alazarmanakelew/.cache/uv/archive-v0/FKWJ9rKDH98yKymc/lib/python3.12/site-packages/transformers/modeling_rope_utils.py:1036: FutureWarning: `rope_config_validation` is deprecated and has been removed. Its functionality has been moved to RotaryEmbeddingConfigMixin.validate_rope method. PreTrainedConfig inherits this class, so please call self.validate_rope() instead. Also, make sure to use the new rope_parameters syntax. You can call self.standardize_rope_params() in the meantime.
  warnings.warn(


PaddleOCR-VL-1.6-4bit  ·  'OCR:'
Reducing Transformer Key-Value Cache Size with Cross-Layer Attention

William Brandon* Mayank Mishra* Aniruddha Nrusimha  MIT CSAIL MIT-IBM Watson AI Lab MIT CSAIL  wbrandon@csail.mit.edu

Rameswar Panda  Jonathan Ragan-Kelley  MIT-IBM Watson AI Lab  MIT CSAIL

Abstract

Key-value (KV) caching plays an essential role in accelerating decoding for transformer-based autoregressive large language models (LLMs).
[746.6 tok/s]


Falcon-OCR  ·  'plain'
# Reducing Transformer Key-Value Cache Size with Cross-Layer Attention

**William Brandon***
MIT CSAIL
wbrandon@csail.mit.edu

**Mayank Mishra***
MIT-IBM Watson AI Lab

**Aniruddha Nrusimha**
MIT CSAIL

**Rameswar Panda**
MIT-IBM Watson AI Lab

**Jonathan Ragan-Kelley**
MIT CSAIL

## Abstract

Key-value (KV) caching plays
[235.3 tok/s]


## Formulas

GLM-OCR returns LaTeX. A 1.2 GB model transcribes an equation as well as a 7B VLM.

In [3]:
run("mlx-community/GLM-OCR-4bit", "Formula Recognition:", FORMULA, max_tokens=80)

GLM-OCR-4bit  ·  'Formula Recognition:'
$$
\mathrm {F F N} (x) = \max \left(0, x W _ {1} + b _ {1}\right) W _ {2} + b _ {2}
$$
[710.5 tok/s]


## Tables

In [4]:
run("mlx-community/GLM-OCR-4bit", "Table Recognition:", TABLE, max_tokens=300)

GLM-OCR-4bit  ·  'Table Recognition:'
BORCELLE

Coffee Shop
MENU

COFFEE
Espresso $5
Double Espresso $5
Latte $5
Americano $5
Macchiato $5
Flat White $5
Cappuccino $5

TEA
Lemon Tea $5
Mango Tea $5
Jasmine $5
Green Tea $5
Mint Tea $5
Hot Chocolate $5
Milkshake $5
Smoothie $5
Lemonade $5
Vanilla Milkshake $5

DESSERTS
Strawberry Waffle $5
Cinnamon Roll $5
Lemon Pie $5
Croissant $5
Chocolate Waffle $5
Brownies $5
Cheesecake $5
Chocolate Muffin $5
[476.8 tok/s]


Which to reach for: PaddleOCR-VL at 0.7 GB for plain text, GLM-OCR for formulas
and tables, Falcon-OCR's layout mode (`generate_with_layout`) for dense
multi-column pages. Each model's full prompt list is in
`mlx_vlm/models/<model_type>/README.md`.